In [13]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

matches = pd.read_csv("../data/processed/matches_clean.csv")
deliveries = pd.read_csv("../data/processed/deliveries_clean.csv")

print(matches.shape, deliveries.shape)

(594, 10) (138834, 12)


In [14]:
wins = matches["winner"].value_counts()
fig = px.bar(x=wins.index, y=wins.values,
             labels={"x": "team", "y": "wins"},
             title="Total wins by team")
fig.show()

In [15]:
matches["toss_win_match_win"] = matches["toss_winner"] == matches["winner"]
pct = matches["toss_win_match_win"].mean() * 100
print(f"Toss winner won the match {pct:.1f}% of the time")

Toss winner won the match 53.7% of the time


In [16]:
decision = matches.groupby("toss_decision")["toss_win_match_win"].mean().reset_index()
decision["toss_win_match_win"] = decision["toss_win_match_win"] * 100
fig = px.bar(decision, x="toss_decision", y="toss_win_match_win",
             title="Win % of toss winner by decision")
fig.show()

In [17]:
runs = deliveries.groupby("batter")["batsman_runs"].sum().nlargest(10).reset_index()
fig = px.bar(runs, x="batter", y="batsman_runs", title="Top 10 run scorers")
fig.show()

In [18]:
merged = deliveries.merge(matches[["id", "season"]], left_on="match_id", right_on="id")
inn1 = merged[merged["inning"] == 1].groupby(["season", "match_id"])["total_runs"].sum().reset_index()
avg = inn1.groupby("season")["total_runs"].mean().reset_index()
fig = px.line(avg, x="season", y="total_runs", markers=True,
              title="Average 1st innings score by season")
fig.show()

In [19]:
deliveries["phase"] = deliveries["over"].apply(lambda o: "Death (16-20)" if o >= 16 else "Overs 1-15")

phase = deliveries.groupby("phase").agg(runs=("total_runs", "sum"), balls=("total_runs", "count")).reset_index()
phase["run_rate"] = phase["runs"] * 6 / phase["balls"]

fig = px.bar(phase, x="phase", y="run_rate", title="Run rate by phase")
fig.show()